# 07 — Pricing and Auction Strategy Simulation

## Plate Value Intelligence

This notebook converts valuation and collectibility outputs into an explainable pricing and auction-routing simulation.

### Objectives

- Combine final holdout valuation predictions with asset-segmentation decisions
- Define transparent commercial routing rules
- Recommend fixed-price, standard-auction, premium-auction, or specialist-review pathways
- Create indicative reserve-price and pricing ranges
- Compare simulated decisions with observed B278 auction outcomes
- Quantify potential commercial opportunities and operational review volumes

### Scope and limitation

This is a decision-support simulation, not a causal pricing-optimisation engine.

The available public dataset does not include bidder histories, bid sequences, listing duration, customer-level demand, price elasticity, or alternative reserve-price experiments. Therefore, the analysis does not claim to identify the revenue-maximising price or auction format.

All strategy rules are transparent, scenario-based, and designed for later calibration with richer internal data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

In [2]:
PROJECT_ROOT = Path.cwd()
REPORTS_DIR = PROJECT_ROOT / "reports"

VALUATION_PREDICTIONS_FILE = (
    REPORTS_DIR
    / "final_valuation_test_predictions_2025.csv"
)

ASSET_DECISIONS_FILE = (
    REPORTS_DIR
    / "premium_collectible_asset_decisions_2025.csv"
)

print(
    f"Valuation predictions exist: "
    f"{VALUATION_PREDICTIONS_FILE.exists()}"
)

print(
    f"Asset decisions exist: "
    f"{ASSET_DECISIONS_FILE.exists()}"
)

Valuation predictions exist: True
Asset decisions exist: True


In [3]:
df_valuation = pd.read_csv(
    VALUATION_PREDICTIONS_FILE
)

df_assets = pd.read_csv(
    ASSET_DECISIONS_FILE
)

print(
    f"Final valuation rows: "
    f"{len(df_valuation):,}"
)

print(
    f"Asset decision rows: "
    f"{len(df_assets):,}"
)

Final valuation rows: 1,981
Asset decision rows: 17,782


## 1. Input Validation

Final holdout valuation predictions are joined to the asset-decision layer using the auction event and lot number as the business key.

In [4]:
required_valuation_columns = [
    "event_code",
    "lot_number",
    "plate_raw",
    "hammer_price_gbp",
    "predicted_price_gbp",
    "residual_gbp",
    "absolute_error_gbp",
]

required_asset_columns = [
    "event_code",
    "lot_number",
    "plate_raw",
    "collectibility_score",
    "collectibility_segment",
    "premium_collectibility_group",
    "specialist_review_required",
    "specialist_review_reason",
    "model_split",
]

missing_valuation_columns = [
    column
    for column in required_valuation_columns
    if column not in df_valuation.columns
]

missing_asset_columns = [
    column
    for column in required_asset_columns
    if column not in df_assets.columns
]

assert not missing_valuation_columns, (
    f"Missing valuation columns: "
    f"{missing_valuation_columns}"
)

assert not missing_asset_columns, (
    f"Missing asset columns: "
    f"{missing_asset_columns}"
)

print("Required pricing-strategy inputs are available.")

Required pricing-strategy inputs are available.


In [5]:
df_assets_test = (
    df_assets.loc[
        df_assets["model_split"].eq("test")
    ]
    .copy()
)

print(
    f"B278 asset rows: "
    f"{len(df_assets_test):,}"
)

B278 asset rows: 1,981


In [6]:
assert len(df_assets_test) == len(df_valuation), (
    "B278 valuation and asset-decision row counts differ."
)

assert df_assets_test["event_code"].eq("B278").all()
assert df_valuation["event_code"].eq("B278").all()

print("Both datasets represent the untouched B278 event.")

Both datasets represent the untouched B278 event.


In [7]:
strategy_data = df_valuation.merge(
    df_assets_test[
        [
            "event_code",
            "lot_number",
            "collectibility_score",
            "collectibility_segment",
            "premium_collectibility_group",
            "specialist_review_required",
            "specialist_review_reason",
        ]
    ],
    on=[
        "event_code",
        "lot_number",
    ],
    how="left",
    validate="one_to_one",
)

In [8]:
print(
    f"Strategy rows: "
    f"{len(strategy_data):,}"
)

print(
    "Missing collectibility decisions: "
    f"{strategy_data['collectibility_score'].isna().sum():,}"
)

Strategy rows: 1,981
Missing collectibility decisions: 0


In [9]:
strategy_data[
    [
        "event_code",
        "lot_number",
        "plate_raw",
        "hammer_price_gbp",
        "predicted_price_gbp",
        "collectibility_score",
        "collectibility_segment",
        "premium_collectibility_group",
        "specialist_review_required",
    ]
].head(10)

,event_code,lot_number,plate_raw,hammer_price_gbp,predicted_price_gbp,collectibility_score,collectibility_segment,premium_collectibility_group,specialist_review_required
0,B278,1,855 A,"10,110.00","10,191.69",40,Collectible,Premium Value Only,1
1,B278,2,9811 A,"4,310.00","5,210.55",13,Standard,Standard Portfolio,0
2,B278,3,3000 AA,"5,010.00","3,562.25",18,Distinctive,Standard Portfolio,0
3,B278,4,A122 AAA,"4,790.00",886.34,18,Distinctive,Standard Portfolio,0
4,B278,5,A154 AAC,"1,510.00",907.53,7,Standard,Standard Portfolio,0
5,B278,6,AAB 10S,"1,010.00","1,913.34",27,Distinctive,Standard Portfolio,0
6,B278,7,AAD 96L,"2,510.00","1,090.30",17,Distinctive,Standard Portfolio,0
7,B278,8,AA54 EED,"1,010.00","1,246.40",17,Distinctive,Standard Portfolio,0
8,B278,9,A400 AGE,350.00,604.00,15,Distinctive,Standard Portfolio,0
9,B278,10,AAM 411D,"1,000.00","1,143.84",10,Standard,Standard Portfolio,0


## 2. Indicative Valuation Range

Point predictions are converted into an indicative valuation range. This range is a transparent scenario band rather than a formal statistical prediction interval.

In [10]:
strategy_data["valuation_low_gbp"] = (
    strategy_data["predicted_price_gbp"]
    * 0.80
)

strategy_data["valuation_central_gbp"] = (
    strategy_data["predicted_price_gbp"]
)

strategy_data["valuation_high_gbp"] = (
    strategy_data["predicted_price_gbp"]
    * 1.25
)

In [11]:
for column in [
    "valuation_low_gbp",
    "valuation_central_gbp",
    "valuation_high_gbp",
]:
    strategy_data[column] = (
        strategy_data[column]
        .round(-1)
    )

In [12]:
strategy_data["predicted_value_tier"] = pd.cut(
    strategy_data["predicted_price_gbp"],
    bins=[
        -np.inf,
        1_000,
        2_500,
        5_000,
        10_000,
        np.inf,
    ],
    labels=[
        "Entry",
        "Core",
        "Upper Core",
        "Premium",
        "Trophy Value",
    ],
)

In [13]:
predicted_value_tier_summary = (
    strategy_data.groupby(
        "predicted_value_tier",
        observed=True,
    )
    .agg(
        plate_count=("plate_raw", "size"),
        median_predicted_price=(
            "predicted_price_gbp",
            "median",
        ),
        median_observed_price=(
            "hammer_price_gbp",
            "median",
        ),
        median_collectibility_score=(
            "collectibility_score",
            "median",
        ),
    )
    .reset_index()
)

predicted_value_tier_summary

,predicted_value_tier,plate_count,median_predicted_price,median_observed_price,median_collectibility_score
0,Entry,662,709.75,640.00,15.00
1,Core,877,"1,434.86","1,500.00",23.00
2,Upper Core,281,"3,104.76","3,520.00",25.00
3,Premium,145,"5,604.39","6,300.00",57.00
4,Trophy Value,16,"10,675.36","10,070.00",38.50


## 3. Commercial Routing Policy

Each registration is assigned to an indicative sales pathway using predicted value, structural collectibility, and specialist-review requirements.

The policy is deliberately transparent and can later be calibrated using internal conversion, bidder, reserve-price, and revenue data.

In [14]:
strategy_data["recommended_sales_path"] = np.select(
    [
        strategy_data[
            "specialist_review_required"
        ].eq(1),

        (
            strategy_data[
                "collectibility_segment"
            ].isin(
                [
                    "High Collectible",
                    "Trophy Candidate",
                ]
            )
            & strategy_data[
                "predicted_price_gbp"
            ].ge(5_000)
        ),

        (
            strategy_data[
                "predicted_price_gbp"
            ].ge(2_500)
        ),
    ],
    [
        "Specialist Review",
        "Premium Auction / Showcase",
        "Standard Auction",
    ],
    default="Fixed Price",
)

In [15]:
def assign_sales_path_reason(row) -> str:
    if row["recommended_sales_path"] == "Specialist Review":
        return row["specialist_review_reason"]

    if (
        row["recommended_sales_path"]
        == "Premium Auction / Showcase"
    ):
        return (
            "high_collectibility_and_high_predicted_value"
        )

    if (
        row["recommended_sales_path"]
        == "Standard Auction"
    ):
        return (
            "meaningful_upside_and_competitive_bidding_potential"
        )

    return (
        "standard_asset_with_limited_auction_upside"
    )


strategy_data["sales_path_reason"] = (
    strategy_data.apply(
        assign_sales_path_reason,
        axis=1,
    )
)

In [16]:
sales_path_summary = (
    strategy_data.groupby(
        "recommended_sales_path"
    )
    .agg(
        plate_count=("plate_raw", "size"),
        median_predicted_price_gbp=(
            "predicted_price_gbp",
            "median",
        ),
        median_observed_price_gbp=(
            "hammer_price_gbp",
            "median",
        ),
        median_collectibility_score=(
            "collectibility_score",
            "median",
        ),
        specialist_review_count=(
            "specialist_review_required",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "median_predicted_price_gbp",
        ascending=False,
    )
)

sales_path_summary["portfolio_share_pct"] = (
    sales_path_summary["plate_count"]
    / len(strategy_data)
    * 100
)

sales_path_summary

,recommended_sales_path,plate_count,median_predicted_price_gbp,median_observed_price_gbp,median_collectibility_score,specialist_review_count,portfolio_share_pct
1,Premium Auction / Showcase,54,"7,240.53","7,005.00",47.00,0,2.73
2,Specialist Review,210,"4,623.96","6,730.00",37.00,210,10.60
3,Standard Auction,235,"3,104.76","3,260.00",25.00,0,11.86
0,Fixed Price,1482,"1,091.21","1,010.00",15.00,0,74.81


## 4. Pricing and Reserve-Price Policy

Indicative prices and reserve levels are assigned using transparent scenario rules.

These rules are not claimed to be revenue-optimal. They are business-policy assumptions designed for retrospective simulation and later calibration with internal bidder, conversion, elasticity, and reserve-price data.

In [17]:
pricing_policy = pd.DataFrame(
    {
        "recommended_sales_path": [
            "Fixed Price",
            "Standard Auction",
            "Premium Auction / Showcase",
            "Specialist Review",
        ],
        "reserve_ratio": [
            np.nan,
            0.70,
            0.65,
            np.nan,
        ],
        "listing_price_ratio": [
            1.10,
            np.nan,
            np.nan,
            np.nan,
        ],
        "policy_note": [
            "List slightly above central valuation to allow pricing headroom.",
            "Use a moderate reserve to protect downside while supporting competition.",
            "Use a lower relative reserve to encourage bidding depth and price discovery.",
            "No automatic price recommendation before specialist review.",
        ],
    }
)

pricing_policy

,recommended_sales_path,reserve_ratio,listing_price_ratio,policy_note
0,Fixed Price,NaN,1.10,List slightly above central valuation to allow pricing headroom.
1,Standard Auction,0.70,NaN,Use a moderate reserve to protect downside while supporting competition.
2,Premium Auction / Showcase,0.65,NaN,Use a lower relative reserve to encourage bidding depth and price discovery.
3,Specialist Review,NaN,NaN,No automatic price recommendation before specialist review.


In [18]:
strategy_data = strategy_data.merge(
    pricing_policy[
        [
            "recommended_sales_path",
            "reserve_ratio",
            "listing_price_ratio",
        ]
    ],
    on="recommended_sales_path",
    how="left",
    validate="many_to_one",
)

In [19]:
strategy_data["recommended_fixed_price_gbp"] = np.where(
    strategy_data["recommended_sales_path"].eq("Fixed Price"),
    (
        strategy_data["valuation_central_gbp"]
        * strategy_data["listing_price_ratio"]
    ),
    np.nan,
)


In [20]:
strategy_data["recommended_reserve_price_gbp"] = np.where(
    strategy_data[
        "recommended_sales_path"
    ].isin(
        [
            "Standard Auction",
            "Premium Auction / Showcase",
        ]
    ),
    (
        strategy_data["valuation_central_gbp"]
        * strategy_data["reserve_ratio"]
    ),
    np.nan,
)

In [21]:
for column in [
    "recommended_fixed_price_gbp",
    "recommended_reserve_price_gbp",
]:
    strategy_data[column] = (
        strategy_data[column]
        .round(-1)
    )

In [22]:
strategy_data[
    [
        "plate_raw",
        "recommended_sales_path",
        "valuation_low_gbp",
        "valuation_central_gbp",
        "valuation_high_gbp",
        "recommended_fixed_price_gbp",
        "recommended_reserve_price_gbp",
    ]
].head(20)

,plate_raw,recommended_sales_path,valuation_low_gbp,valuation_central_gbp,valuation_high_gbp,recommended_fixed_price_gbp,recommended_reserve_price_gbp
0,855 A,Specialist Review,"8,150.00","10,190.00","12,740.00",NaN,NaN
1,9811 A,Standard Auction,"4,170.00","5,210.00","6,510.00",NaN,"3,650.00"
2,3000 AA,Standard Auction,"2,850.00","3,560.00","4,450.00",NaN,"2,490.00"
3,A122 AAA,Fixed Price,710.00,890.00,"1,110.00",980.00,NaN
4,A154 AAC,Fixed Price,730.00,910.00,"1,130.00","1,000.00",NaN
5,AAB 10S,Fixed Price,"1,530.00","1,910.00","2,390.00","2,100.00",NaN
6,AAD 96L,Fixed Price,870.00,"1,090.00","1,360.00","1,200.00",NaN
7,AA54 EED,Fixed Price,"1,000.00","1,250.00","1,560.00","1,380.00",NaN
8,A400 AGE,Fixed Price,480.00,600.00,760.00,660.00,NaN
9,AAM 411D,Fixed Price,920.00,"1,140.00","1,430.00","1,250.00",NaN


In [23]:
specialist_mask = (
    strategy_data[
        "recommended_sales_path"
    ].eq("Specialist Review")
)

strategy_data.loc[
    specialist_mask,
    [
        "recommended_fixed_price_gbp",
        "recommended_reserve_price_gbp",
    ],
] = np.nan

In [24]:
assert strategy_data.loc[
    specialist_mask,
    "recommended_fixed_price_gbp",
].isna().all()

assert strategy_data.loc[
    specialist_mask,
    "recommended_reserve_price_gbp",
].isna().all()

print(
    "Specialist-review assets have no automatic "
    "listing or reserve price."
)

Specialist-review assets have no automatic listing or reserve price.


## 5. Retrospective Reserve Coverage

Observed hammer prices are compared with the simulated reserve recommendations.

This analysis does not show what would have happened under a different reserve. It only checks whether the indicative reserve would have been below the observed hammer price.

In [25]:
auction_mask = (
    strategy_data[
        "recommended_sales_path"
    ].isin(
        [
            "Standard Auction",
            "Premium Auction / Showcase",
        ]
    )
)

strategy_data["observed_price_above_reserve"] = np.where(
    auction_mask,
    (
        strategy_data["hammer_price_gbp"]
        >= strategy_data[
            "recommended_reserve_price_gbp"
        ]
    ).astype(int),
    np.nan,
)

In [26]:
reserve_coverage_summary = (
    strategy_data.loc[auction_mask]
    .groupby("recommended_sales_path")
    .agg(
        plate_count=("plate_raw", "size"),
        median_observed_price_gbp=(
            "hammer_price_gbp",
            "median",
        ),
        median_reserve_price_gbp=(
            "recommended_reserve_price_gbp",
            "median",
        ),
        observed_price_above_reserve_rate=(
            "observed_price_above_reserve",
            "mean",
        ),
    )
    .reset_index()
)

reserve_coverage_summary[
    "observed_price_above_reserve_rate_pct"
] = (
    reserve_coverage_summary[
        "observed_price_above_reserve_rate"
    ]
    * 100
)

reserve_coverage_summary

,recommended_sales_path,plate_count,median_observed_price_gbp,median_reserve_price_gbp,observed_price_above_reserve_rate,observed_price_above_reserve_rate_pct
0,Premium Auction / Showcase,54,"7,005.00","4,710.00",1.00,100.00
1,Standard Auction,235,"3,260.00","2,170.00",0.76,75.74


In [27]:
fixed_price_mask = (
    strategy_data[
        "recommended_sales_path"
    ].eq("Fixed Price")
)

strategy_data["observed_vs_recommended_fixed_gap_gbp"] = np.where(
    fixed_price_mask,
    (
        strategy_data["hammer_price_gbp"]
        - strategy_data[
            "recommended_fixed_price_gbp"
        ]
    ),
    np.nan,
)

In [28]:
fixed_price_summary = pd.DataFrame(
    {
        "plate_count": [
            int(fixed_price_mask.sum())
        ],
        "median_observed_price_gbp": [
            strategy_data.loc[
                fixed_price_mask,
                "hammer_price_gbp",
            ].median()
        ],
        "median_recommended_fixed_price_gbp": [
            strategy_data.loc[
                fixed_price_mask,
                "recommended_fixed_price_gbp",
            ].median()
        ],
        "median_observed_minus_recommended_gbp": [
            strategy_data.loc[
                fixed_price_mask,
                "observed_vs_recommended_fixed_gap_gbp",
            ].median()
        ],
    }
)

fixed_price_summary

,plate_count,median_observed_price_gbp,median_recommended_fixed_price_gbp,median_observed_minus_recommended_gbp
0,1482,"1,010.00","1,200.00",-190.00


In [29]:
strategy_calibration_summary = (
    strategy_data.groupby(
        "recommended_sales_path"
    )
    .agg(
        plate_count=("plate_raw", "size"),
        observed_total_value_gbp=(
            "hammer_price_gbp",
            "sum",
        ),
        median_observed_price_gbp=(
            "hammer_price_gbp",
            "median",
        ),
        median_predicted_price_gbp=(
            "predicted_price_gbp",
            "median",
        ),
        median_absolute_error_gbp=(
            "absolute_error_gbp",
            "median",
        ),
        median_collectibility_score=(
            "collectibility_score",
            "median",
        ),
    )
    .reset_index()
)

strategy_calibration_summary[
    "portfolio_share_pct"
] = (
    strategy_calibration_summary[
        "plate_count"
    ]
    / len(strategy_data)
    * 100
)

strategy_calibration_summary.sort_values(
    "median_observed_price_gbp",
    ascending=False,
)

,recommended_sales_path,plate_count,observed_total_value_gbp,median_observed_price_gbp,median_predicted_price_gbp,median_absolute_error_gbp,median_collectibility_score,portfolio_share_pct
1,Premium Auction / Showcase,54,"454,280.00","7,005.00","7,240.53","1,096.54",47.00,2.73
2,Specialist Review,210,"1,751,180.00","6,730.00","4,623.96","3,292.68",37.00,10.60
3,Standard Auction,235,"813,710.00","3,260.00","3,104.76",946.95,25.00,11.86
0,Fixed Price,1482,"2,076,650.00","1,010.00","1,091.21",495.26,15.00,74.81


## 6. Commercial Opportunity Flags

Differences between predicted and observed values are used as retrospective investigation signals, not as causal estimates of missed revenue.

In [30]:
strategy_data["valuation_gap_gbp"] = (
    strategy_data["predicted_price_gbp"]
    - strategy_data["hammer_price_gbp"]
)

strategy_data["valuation_gap_pct"] = (
    strategy_data["valuation_gap_gbp"]
    / strategy_data["hammer_price_gbp"]
    * 100
)

In [31]:
strategy_data["commercial_opportunity_flag"] = np.select(
    [
        (
            strategy_data[
                "premium_collectibility_group"
            ].eq("Collectible Potential")
            & strategy_data[
                "predicted_price_gbp"
            ].gt(
                strategy_data[
                    "hammer_price_gbp"
                ]
            )
        ),
        (
            strategy_data[
                "recommended_sales_path"
            ].eq("Specialist Review")
        ),
        (
            strategy_data[
                "predicted_price_gbp"
            ]
            >= strategy_data[
                "hammer_price_gbp"
            ] * 1.50
        ),
    ],
    [
        "collectible_value_review",
        "specialist_assessment",
        "material_valuation_gap_review",
    ],
    default="standard_monitoring",
)

In [32]:
commercial_opportunity_summary = (
    strategy_data.groupby(
        "commercial_opportunity_flag"
    )
    .agg(
        plate_count=("plate_raw", "size"),
        median_observed_price_gbp=(
            "hammer_price_gbp",
            "median",
        ),
        median_predicted_price_gbp=(
            "predicted_price_gbp",
            "median",
        ),
        median_collectibility_score=(
            "collectibility_score",
            "median",
        ),
    )
    .reset_index()
    .sort_values(
        "plate_count",
        ascending=False,
    )
)

commercial_opportunity_summary

,commercial_opportunity_flag,plate_count,median_observed_price_gbp,median_predicted_price_gbp,median_collectibility_score
3,standard_monitoring,1209,"1,950.00","1,255.05",17.00
1,material_valuation_gap_review,562,410.00,"1,113.48",15.00
2,specialist_assessment,164,"7,515.00","3,292.50",30.00
0,collectible_value_review,46,"4,550.00","5,432.52",57.00


In [33]:
pricing_strategy_decisions = strategy_data[
    [
        "event_code",
        "lot_number",
        "plate_raw",
        "hammer_price_gbp",
        "predicted_price_gbp",
        "valuation_low_gbp",
        "valuation_central_gbp",
        "valuation_high_gbp",
        "predicted_value_tier",
        "collectibility_score",
        "collectibility_segment",
        "premium_collectibility_group",
        "recommended_sales_path",
        "sales_path_reason",
        "recommended_fixed_price_gbp",
        "recommended_reserve_price_gbp",
        "specialist_review_required",
        "specialist_review_reason",
        "commercial_opportunity_flag",
    ]
].copy()

pricing_strategy_decisions.head(20)

,event_code,lot_number,plate_raw,hammer_price_gbp,predicted_price_gbp,valuation_low_gbp,valuation_central_gbp,valuation_high_gbp,predicted_value_tier,collectibility_score,collectibility_segment,premium_collectibility_group,recommended_sales_path,sales_path_reason,recommended_fixed_price_gbp,recommended_reserve_price_gbp,specialist_review_required,specialist_review_reason,commercial_opportunity_flag
0,B278,1,855 A,"10,110.00","10,191.69","8,150.00","10,190.00","12,740.00",Trophy Value,40,Collectible,Premium Value Only,Specialist Review,high_market_value_not_fully_explained,NaN,NaN,1,high_market_value_not_fully_explained,specialist_assessment
1,B278,2,9811 A,"4,310.00","5,210.55","4,170.00","5,210.00","6,510.00",Premium,13,Standard,Standard Portfolio,Standard Auction,meaningful_upside_and_competitive_bidding_potential,NaN,"3,650.00",0,standard_automated_path,standard_monitoring
2,B278,3,3000 AA,"5,010.00","3,562.25","2,850.00","3,560.00","4,450.00",Upper Core,18,Distinctive,Standard Portfolio,Standard Auction,meaningful_upside_and_competitive_bidding_potential,NaN,"2,490.00",0,standard_automated_path,standard_monitoring
3,B278,4,A122 AAA,"4,790.00",886.34,710.00,890.00,"1,110.00",Entry,18,Distinctive,Standard Portfolio,Fixed Price,standard_asset_with_limited_auction_upside,980.00,NaN,0,standard_automated_path,standard_monitoring
4,B278,5,A154 AAC,"1,510.00",907.53,730.00,910.00,"1,130.00",Entry,7,Standard,Standard Portfolio,Fixed Price,standard_asset_with_limited_auction_upside,"1,000.00",NaN,0,standard_automated_path,standard_monitoring
5,B278,6,AAB 10S,"1,010.00","1,913.34","1,530.00","1,910.00","2,390.00",Core,27,Distinctive,Standard Portfolio,Fixed Price,standard_asset_with_limited_auction_upside,"2,100.00",NaN,0,standard_automated_path,material_valuation_gap_review
6,B278,7,AAD 96L,"2,510.00","1,090.30",870.00,"1,090.00","1,360.00",Core,17,Distinctive,Standard Portfolio,Fixed Price,standard_asset_with_limited_auction_upside,"1,200.00",NaN,0,standard_automated_path,standard_monitoring
7,B278,8,AA54 EED,"1,010.00","1,246.40","1,000.00","1,250.00","1,560.00",Core,17,Distinctive,Standard Portfolio,Fixed Price,standard_asset_with_limited_auction_upside,"1,380.00",NaN,0,standard_automated_path,standard_monitoring
8,B278,9,A400 AGE,350.00,604.00,480.00,600.00,760.00,Entry,15,Distinctive,Standard Portfolio,Fixed Price,standard_asset_with_limited_auction_upside,660.00,NaN,0,standard_automated_path,material_valuation_gap_review
9,B278,10,AAM 411D,"1,000.00","1,143.84",920.00,"1,140.00","1,430.00",Core,10,Standard,Standard Portfolio,Fixed Price,standard_asset_with_limited_auction_upside,"1,250.00",NaN,0,standard_automated_path,standard_monitoring
